In [12]:
import pyspark
from pyspark.sql import SparkSession

In [16]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Spark version: 4.1.1
JVM Java version: 25.0.1


In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 15:22:40--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.181, 18.239.38.147, 18.239.38.83, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.181|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   271MB/s    in 0.3s    

2026-03-09 15:22:41 (271 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [15]:

df_yellow_2025_11 = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df_yellow_2025_11.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [16]:
df_yellow_2025_11.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [17]:
df_yellow_2025_11.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [18]:
df_yellow_2025_11.registerTempTable('trips_data')

/workspaces/Zoomcamp_DE_2026_Homework/6_Batching/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [20]:
spark.sql("""
SELECT
    COUNT(*) AS trips_count
FROM
    trips_data
WHERE tpep_pickup_datetime >= '2025-11-15' AND tpep_pickup_datetime < '2025-11-16'
""").show()

+-----------+
|trips_count|
+-----------+
|     162604|
+-----------+



In [28]:
spark.sql("""
SELECT
    date_trunc('day', tpep_pickup_datetime) AS pickup_date,
    COUNT(*) AS trips_count
FROM
    trips_data
GROUP BY date_trunc('day', tpep_pickup_datetime)
Order BY pickup_date
""").show()

+-------------------+-----------+
|        pickup_date|trips_count|
+-------------------+-----------+
|2008-12-31 00:00:00|          1|
|2009-01-01 00:00:00|          2|
|2025-10-31 00:00:00|         21|
|2025-11-01 00:00:00|     183032|
|2025-11-02 00:00:00|     152130|
|2025-11-03 00:00:00|     129816|
|2025-11-04 00:00:00|     131512|
|2025-11-05 00:00:00|     142587|
|2025-11-06 00:00:00|     158170|
|2025-11-07 00:00:00|     147795|
|2025-11-08 00:00:00|     153432|
|2025-11-09 00:00:00|     133197|
|2025-11-10 00:00:00|     131860|
|2025-11-11 00:00:00|     131252|
|2025-11-12 00:00:00|     142267|
|2025-11-13 00:00:00|     156277|
|2025-11-14 00:00:00|     154779|
|2025-11-15 00:00:00|     162604|
|2025-11-16 00:00:00|     136369|
|2025-11-17 00:00:00|     130552|
+-------------------+-----------+
only showing top 20 rows
